# Sample Geometry Diagnostics

This notebook prepares final report diagnostics in the sample-geometry and distribution figures. 
It compares real and generated path samples through financial feature vectors rather than through raw path co-ordinates.

The t-SNE panels are qualitative visual checks only. Wider or tighter spread in a t-SNE panel is not, by itself, evidence of better financial generation. Distributional retention should be read primarily through KDE/ECDF feature overlays and the quantitative metrics recorded in the model registry and model cards.


In [ ]:
EXPERIMENTS = ["sp500_vix", "hawkes_jump"]
AVAILABLE_EXPERIMENTS = ["black_scholes", "heston", "pdv", "sp500_vix", "hawkes_jump"]

MODEL_FAMILIES = ["continuous", "discrete"]
USE_REGISTRY_DEFAULTS = True
INCLUDE_OPTIONAL_RESEARCH_MODELS = False

RUN_TSNE = True
RUN_KDE = True
RUN_FULL = False
ALLOW_MISSING_OUTPUTS = True

MODEL_REGISTRY_PATH = "../../trained_models/model_registry.yaml"
OUTPUT_DIR = "../../outputs/final_sample_geometry_report"

# Optional manual overrides:
BATCH_PATH_OVERRIDES = {
    # "sp500_vix": {
    #   "real": ".../evaluation_batch.pt",
    #   "continuous": ".../evaluation_batch.pt",
    #   "discrete": ".../evaluation_batch.pt",
    # }
}

## Setup

The notebook resolves paths relative to the repository root and to this notebook's directory, so the parameter paths above work both under `jupyter nbconvert` from the repository root and in an interactive notebook session.


In [ ]:
import importlib.util
import json
import math
import sys
from collections.abc import Mapping
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "report"
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from time_causal_vae.evaluation.sample_geometry import (
    PathFeatureMatrix,
    fit_tsne_projection,
    kde_or_ecdf_summary,
    path_feature_matrix,
)
from time_causal_vae.experiments.model_registry import (
    FAMILIES,
    load_registry,
    select_registered_model,
)

SKLEARN_AVAILABLE = importlib.util.find_spec("sklearn") is not None
SCIPY_AVAILABLE = importlib.util.find_spec("scipy") is not None
LIGHTWEIGHT_SAMPLE_COUNT = 250
PLOT_SAMPLE_COUNT = None if RUN_FULL else LIGHTWEIGHT_SAMPLE_COUNT
RANDOM_SEED = 0

torch.manual_seed(RANDOM_SEED)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180})

print(f"Repository root: {REPO_ROOT}")
print(f"sklearn available: {SKLEARN_AVAILABLE}")
print(f"scipy available: {SCIPY_AVAILABLE}")
print(f"plot sample cap: {PLOT_SAMPLE_COUNT if PLOT_SAMPLE_COUNT else 'full batches'}")

In [ ]:
def resolve_parameter_path(raw_path: str | Path, *, must_exist: bool = False) -> Path:
    path = Path(raw_path).expanduser()
    if path.is_absolute():
        if must_exist and not path.exists():
            raise FileNotFoundError(path)
        return path
    candidates = [
        (NOTEBOOK_DIR / path).resolve(),
        (REPO_ROOT / path).resolve(),
        (Path.cwd() / path).resolve(),
    ]
    if must_exist:
        for candidate in candidates:
            if candidate.exists():
                return candidate
        raise FileNotFoundError(f"Could not resolve {raw_path!r}; tried {candidates}")
    return candidates[0]


def relative_to_repo(path: Path | None) -> str | None:
    if path is None:
        return None
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


REGISTRY_PATH = resolve_parameter_path(MODEL_REGISTRY_PATH, must_exist=True)
REPORT_OUTPUT_DIR = resolve_parameter_path(OUTPUT_DIR)
OUTPUT_ROOT = REPO_ROOT / "outputs"
REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Registry: {relative_to_repo(REGISTRY_PATH)}")
print(f"Report output directory: {relative_to_repo(REPORT_OUTPUT_DIR)}")

## Registry Selection

The notebook uses the metadata registry to report the selected continuous and discrete candidates. It does not evaluate checkpoints or change model selections. Optional research candidates are listed only when `INCLUDE_OPTIONAL_RESEARCH_MODELS` is enabled.


In [ ]:
registry = load_registry(REGISTRY_PATH)
registered_experiments = set((registry.get("experiments") or {}).keys())
requested_experiments = [
    experiment for experiment in EXPERIMENTS if experiment in AVAILABLE_EXPERIMENTS
]
unknown_experiments = [
    experiment for experiment in EXPERIMENTS if experiment not in AVAILABLE_EXPERIMENTS
]
if unknown_experiments:
    display(Markdown(f"Ignoring unknown experiment names: `{unknown_experiments}`."))

missing_registry_experiments = [
    experiment for experiment in requested_experiments if experiment not in registered_experiments
]
if missing_registry_experiments:
    display(
        Markdown(
            "Registry entries are missing for: "
            + ", ".join(f"`{experiment}`" for experiment in missing_registry_experiments)
        )
    )


def candidate_mapping(experiment: str, family: str) -> Mapping[str, Mapping[str, Any]]:
    section = registry.get("experiments", {}).get(experiment, {}).get(family, {})
    candidates = section.get("candidates", {}) if isinstance(section, Mapping) else {}
    return candidates if isinstance(candidates, Mapping) else {}


def iter_requested_model_records() -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for experiment in requested_experiments:
        if experiment not in registered_experiments:
            continue
        for family in MODEL_FAMILIES:
            if family not in FAMILIES:
                continue
            try:
                selected = select_registered_model(
                    registry,
                    experiment,
                    family,  # type: ignore[arg-type]
                )
            except Exception as exc:
                records.append({
                    "experiment": experiment,
                    "family": family,
                    "candidate_id": None,
                    "status": "missing_registry_family",
                    "note": str(exc),
                    "registered_model": None,
                })
                continue
            records.append({
                "experiment": experiment,
                "family": family,
                "candidate_id": selected.candidate_id,
                "status": "selected",
                "note": selected.selected_by,
                "registered_model": selected,
            })
            if INCLUDE_OPTIONAL_RESEARCH_MODELS:
                for candidate_id, candidate in candidate_mapping(experiment, family).items():
                    if candidate_id == selected.candidate_id:
                        continue
                    status = str(candidate.get("status", ""))
                    is_optional = (
                        "research" in status
                        or candidate.get("public_default") is False
                        or candidate.get("selected") is False
                    )
                    if is_optional:
                        records.append({
                            "experiment": experiment,
                            "family": family,
                            "candidate_id": str(candidate_id),
                            "status": "optional_research",
                            "note": status or "optional candidate",
                            "registered_model": None,
                            "candidate": dict(candidate),
                        })
    return records


model_records = iter_requested_model_records()
registry_rows = []
for record in model_records:
    selected = record.get("registered_model")
    candidate = getattr(selected, "candidate", record.get("candidate", {})) or {}
    registry_rows.append({
        "experiment": record["experiment"],
        "family": record["family"],
        "candidate_id": record["candidate_id"],
        "status": record["status"],
        "selection": record["note"],
        "selection_profile": getattr(
            selected, "selection_profile", candidate.get("selection_profile")
        ),
        "checkpoint_convention": candidate.get("checkpoint_convention"),
        "metrics_available": len(getattr(selected, "metrics", candidate.get("metrics", {})) or {}),
    })

registry_table = pd.DataFrame(registry_rows)
display(registry_table)

## Local Batch Discovery

Local batches are optional. The notebook searches common `outputs/` locations for existing path payloads and summary JSON files, then applies `BATCH_PATH_OVERRIDES` when provided. Missing outputs are reported rather than treated as notebook failures.


In [ ]:
BATCH_FILE_NAMES = {
    "evaluation_batch.pt",
    "score_prior_samples.pt",
    "decoded_paths.pt",
    "discrete_paper_style_batch.pt",
    "continuous_paper_style_batch.pt",
}
SUMMARY_FILE_NAMES = {
    "evaluation_summary.json",
    "score_prior_evaluation_summary.json",
    "summary.json",
    "paper_style_summary.json",
    "token_prior_summary.json",
    "comparison_manifest.json",
}
REAL_KEYS = (
    "real_paths",
    "real_data",
    "real_decoder_space",
    "paths",
    "data",
    "batch",
    "samples",
)
GENERATED_KEYS = (
    "decoded_paths",
    "generated_paths",
    "fake_paths",
    "fake_data",
    "generated_decoder_space",
    "decoded_decoder_space",
    "paths",
    "samples",
    "data",
    "batch",
)
FEATURE_ORDER = [
    "terminal_return",
    "realised_volatility",
    "maximum_drawdown",
    "return_autocorr_lag_1",
    "squared_return_autocorr_lag_1",
    "detected_jump_count",
    "return_var_01",
    "return_expected_shortfall_01",
]


def collect_files(root: Path, names: set[str]) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.rglob("*") if path.is_file() and path.name in names)


batch_candidates = collect_files(OUTPUT_ROOT, BATCH_FILE_NAMES)
summary_candidates = collect_files(OUTPUT_ROOT, SUMMARY_FILE_NAMES)
print(f"Candidate batch files found: {len(batch_candidates)}")
print(f"Candidate summary files found: {len(summary_candidates)}")


def tokenise_identifier(value: str) -> set[str]:
    tokens = value.replace("-", "_").replace("/", "_").split("_")
    return {token for token in tokens if token and token not in {"vq", "ar", "causal"}}


def score_batch_path(path: Path, *, experiment: str, family: str, candidate_id: str | None) -> int:
    text = relative_to_repo(path) or str(path)
    lowered = text.lower()
    score = 0
    if experiment.lower() in lowered:
        score += 50
    else:
        return -10_000
    if family == "continuous":
        if "continuous" in lowered:
            score += 35
        if "legacy_continuous" in lowered:
            score += 30
        if path.name == "continuous_paper_style_batch.pt":
            score += 25
        if path.name == "evaluation_batch.pt":
            score += 20
        if "discrete" in lowered or "token_prior" in lowered:
            score -= 40
    elif family == "discrete":
        if "discrete" in lowered or "token_prior" in lowered or "path_metrics" in lowered:
            score += 35
        if path.name == "discrete_paper_style_batch.pt":
            score += 30
        if path.name == "decoded_paths.pt":
            score += 25
        if path.name == "evaluation_batch.pt":
            score += 15
        if "continuous" in lowered and "hawkes_jump_logreturn_robustness" not in lowered:
            score -= 35
    if candidate_id:
        candidate_tokens = tokenise_identifier(candidate_id)
        path_tokens = tokenise_identifier(lowered)
        score += 4 * len(candidate_tokens & path_tokens)
    if "seed0" in lowered or "evaluation_final" in lowered:
        score += 4
    if "smoke" in lowered or "dry" in lowered:
        score -= 10
    return score


def override_path(experiment: str, key: str) -> Path | None:
    value = BATCH_PATH_OVERRIDES.get(experiment, {}).get(key)
    if value is None:
        return None
    return resolve_parameter_path(value, must_exist=False)


def discover_batch_path(
    experiment: str, family: str, candidate_id: str | None
) -> tuple[Path | None, str]:
    manual = override_path(experiment, family)
    if manual is not None:
        return manual if manual.exists() else None, f"manual override: {relative_to_repo(manual)}"
    scored = sorted(
        (
            (
                score_batch_path(
                    path, experiment=experiment, family=family, candidate_id=candidate_id
                ),
                path,
            )
            for path in batch_candidates
        ),
        key=lambda item: (item[0], str(item[1])),
        reverse=True,
    )
    if scored and scored[0][0] > 0:
        return scored[0][1], f"auto-discovered score={scored[0][0]}"
    return None, "no local batch matched the experiment/family"


def find_summary_for_path(batch_path: Path | None) -> Path | None:
    if batch_path is None:
        return None
    preferred = [
        batch_path.with_name("evaluation_summary.json"),
        batch_path.with_name("score_prior_evaluation_summary.json"),
        batch_path.with_name("summary.json"),
        batch_path.with_name("paper_style_summary.json"),
        batch_path.with_name("token_prior_summary.json"),
        batch_path.with_name("comparison_manifest.json"),
    ]
    for candidate in preferred:
        if candidate.exists():
            return candidate
    same_dir = [
        candidate for candidate in summary_candidates if candidate.parent == batch_path.parent
    ]
    return same_dir[0] if same_dir else None


def load_summary_metrics(summary_path: Path | None) -> dict[str, float]:
    if summary_path is None or not summary_path.exists():
        return {}
    try:
        payload = json.loads(summary_path.read_text())
    except Exception:
        return {}

    def flatten_numeric(prefix: str, value: Any, out: dict[str, float]) -> None:
        if isinstance(value, bool):
            return
        if isinstance(value, int | float) and math.isfinite(float(value)):
            out[prefix] = float(value)
        elif isinstance(value, Mapping):
            for key, item in value.items():
                name = f"{prefix}.{key}" if prefix else str(key)
                flatten_numeric(name, item, out)

    flat: dict[str, float] = {}
    flatten_numeric("", payload, flat)
    preferred = [
        key
        for key in flat
        if any(
            token in key
            for token in [
                "mmd",
                "swd",
                "wasserstein",
                "return_ac",
                "squared_return_ac",
                "jump_count",
                "var_01",
                "es_01",
            ]
        )
    ]
    return {key: flat[key] for key in sorted(preferred)[:20]}


batch_rows = []
missing_rows = []
for record in model_records:
    if record.get("candidate_id") is None:
        missing_rows.append({
            "experiment": record["experiment"],
            "family": record["family"],
            "candidate_id": None,
            "reason": record.get("note", "missing registry entry"),
            "note": "Add or repair the registry entry before plotting this model.",
        })
        continue
    batch_path, discovery_note = discover_batch_path(
        record["experiment"],
        record["family"],
        record["candidate_id"],
    )
    real_override = override_path(record["experiment"], "real")
    real_path = (
        real_override if real_override is not None and real_override.exists() else batch_path
    )
    summary_path = find_summary_for_path(batch_path)
    record["batch_path"] = batch_path
    record["real_path"] = real_path
    record["summary_path"] = summary_path
    record["discovery_note"] = discovery_note
    batch_rows.append({
        "experiment": record["experiment"],
        "family": record["family"],
        "candidate_id": record["candidate_id"],
        "batch_path": relative_to_repo(batch_path),
        "real_path": relative_to_repo(real_path),
        "summary_path": relative_to_repo(summary_path),
        "discovery": discovery_note,
    })
    if batch_path is None:
        missing_rows.append({
            "experiment": record["experiment"],
            "family": record["family"],
            "candidate_id": record["candidate_id"],
            "reason": "missing local batch",
            "note": (
                "Set BATCH_PATH_OVERRIDES or create a local evaluation batch; "
                "this notebook does not evaluate checkpoints by default."
            ),
        })

batch_table = pd.DataFrame(batch_rows)
missing_outputs_table = pd.DataFrame(missing_rows)
display(batch_table)
if not missing_outputs_table.empty:
    display(Markdown("### Missing or skipped outputs"))
    display(missing_outputs_table)

## Load Feature Matrices

Each available model comparison contributes a real feature matrix and one generated feature matrix. With `RUN_FULL=False`, the notebook uses a deterministic subsample for plotting and table generation so lightweight execution remains fast.


In [ ]:
def dataset_type_for_experiment(experiment: str) -> str:
    if experiment == "sp500_vix":
        return "sp500_vix"
    if experiment == "hawkes_jump":
        return "hawkes_jump"
    return "generic"


def load_tensor_from_payload(path: Path, *, role: str) -> torch.Tensor:
    preferred = REAL_KEYS if role == "real" else GENERATED_KEYS
    payload = torch.load(path, map_location="cpu")
    if isinstance(payload, torch.Tensor):
        return payload.detach().float()
    if not isinstance(payload, Mapping):
        raise ValueError(f"Expected tensor or mapping payload in {relative_to_repo(path)}")
    for key in preferred:
        value = payload.get(key)
        if isinstance(value, torch.Tensor):
            return value.detach().float()
    tensor_candidates = [
        value.detach().float()
        for value in payload.values()
        if isinstance(value, torch.Tensor) and value.ndim in {2, 3}
    ]
    if len(tensor_candidates) == 1:
        return tensor_candidates[0]
    keys = ", ".join(str(key) for key in payload)
    raise ValueError(f"Could not select {role} tensor from {relative_to_repo(path)}. Keys: {keys}")


def deterministic_subsample(
    paths: torch.Tensor, *, max_samples: int | None, seed: int
) -> torch.Tensor:
    if max_samples is None or paths.shape[0] <= max_samples:
        return paths
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(paths.shape[0], generator=generator)[:max_samples]
    return paths.index_select(0, indices)


def feature_stats_rows(
    experiment: str,
    family: str,
    candidate_id: str,
    source: str,
    features: PathFeatureMatrix,
) -> list[dict[str, Any]]:
    rows = []
    selected_features = [name for name in FEATURE_ORDER if name in features.feature_names]
    for name in selected_features:
        column = features.values[:, features.feature_names.index(name)]
        rows.append({
            "experiment": experiment,
            "family": family,
            "candidate_id": candidate_id,
            "source": source,
            "feature": name,
            "mean": float(column.mean().item()),
            "std": float(column.std(unbiased=False).item()),
        })
    return rows


panel_records = []
feature_stat_rows = []
load_failures = []
for record in model_records:
    batch_path = record.get("batch_path")
    real_path = record.get("real_path")
    candidate_id = record.get("candidate_id")
    if batch_path is None or real_path is None or candidate_id is None:
        continue
    try:
        real_paths = load_tensor_from_payload(real_path, role="real")
        generated_paths = load_tensor_from_payload(batch_path, role="generated")
        real_paths = deterministic_subsample(
            real_paths,
            max_samples=PLOT_SAMPLE_COUNT,
            seed=RANDOM_SEED,
        )
        generated_paths = deterministic_subsample(
            generated_paths,
            max_samples=PLOT_SAMPLE_COUNT,
            seed=RANDOM_SEED + 1,
        )
        dataset_type = dataset_type_for_experiment(record["experiment"])
        real_features = path_feature_matrix(real_paths, dataset_type)  # type: ignore[arg-type]
        generated_features = path_feature_matrix(generated_paths, dataset_type)  # type: ignore[arg-type]
    except Exception as exc:
        load_failures.append({
            "experiment": record["experiment"],
            "family": record["family"],
            "candidate_id": candidate_id,
            "batch_path": relative_to_repo(batch_path),
            "reason": str(exc),
        })
        continue

    label = f"{record['family']}: {candidate_id}"
    panel = {
        "experiment": record["experiment"],
        "family": record["family"],
        "candidate_id": candidate_id,
        "label": label,
        "dataset_type": dataset_type,
        "real_features": real_features,
        "generated_features": generated_features,
        "batch_path": batch_path,
        "real_path": real_path,
        "summary_path": record.get("summary_path"),
    }
    panel_records.append(panel)
    feature_stat_rows.extend(
        feature_stats_rows(
            record["experiment"],
            record["family"],
            candidate_id,
            "real",
            real_features,
        )
    )
    feature_stat_rows.extend(
        feature_stats_rows(
            record["experiment"],
            record["family"],
            candidate_id,
            "generated",
            generated_features,
        )
    )

feature_stats_table = pd.DataFrame(feature_stat_rows)
load_failures_table = pd.DataFrame(load_failures)
print(f"Loaded model comparisons: {len(panel_records)}")
if not load_failures_table.empty:
    display(Markdown("### Batch load failures"))
    display(load_failures_table)
display(feature_stats_table.head(40) if not feature_stats_table.empty else feature_stats_table)

## t-SNE Sample Geometry

The grid overlays real and generated samples after converting each path into a standardised financial feature vector. The implementation uses scikit-learn t-SNE when available and records a PCA fallback otherwise.


In [ ]:
def plot_projection_axis(axis: Any, panel: Mapping[str, Any]) -> str:
    projection = fit_tsne_projection(
        panel["real_features"],
        {panel["family"]: panel["generated_features"]},
        random_state=RANDOM_SEED,
    )
    coordinates = projection.coordinates.detach().cpu()
    labels = projection.labels
    for label in dict.fromkeys(labels):
        indices = [index for index, observed in enumerate(labels) if observed == label]
        points = coordinates[indices]
        axis.scatter(points[:, 0], points[:, 1], s=12, alpha=0.72, label=label)
    axis.set_xlabel("component 1")
    axis.set_ylabel("component 2")
    axis.legend(frameon=False, fontsize="x-small")
    return projection.method


tsne_path = REPORT_OUTPUT_DIR / "tsne_grid.png"
if RUN_TSNE and panel_records:
    row_families = [
        family for family in MODEL_FAMILIES if any(p["family"] == family for p in panel_records)
    ]
    col_experiments = [
        experiment
        for experiment in requested_experiments
        if any(p["experiment"] == experiment for p in panel_records)
    ]
    figure, axes = plt.subplots(
        len(row_families),
        len(col_experiments),
        figsize=(4.8 * len(col_experiments), 4.0 * len(row_families)),
        squeeze=False,
        constrained_layout=True,
    )
    projection_rows = []
    for row_index, family in enumerate(row_families):
        for col_index, experiment in enumerate(col_experiments):
            axis = axes[row_index][col_index]
            matching = [
                panel
                for panel in panel_records
                if panel["family"] == family and panel["experiment"] == experiment
            ]
            if not matching:
                axis.set_axis_off()
                axis.set_title(f"{experiment} / {family}: missing")
                continue
            panel = matching[0]
            method = plot_projection_axis(axis, panel)
            axis.set_title(f"{experiment} / {family} ({method})")
            projection_rows.append({
                "experiment": experiment,
                "family": family,
                "candidate_id": panel["candidate_id"],
                "projection_method": method,
            })
    figure.suptitle("Qualitative feature-space sample geometry", y=1.02)
    figure.savefig(tsne_path)
    plt.close(figure)
    display(pd.DataFrame(projection_rows))
    display(Markdown(f"Saved t-SNE/PCA grid to `{relative_to_repo(tsne_path)}`."))
elif RUN_TSNE:
    display(Markdown("No local model comparisons were available for the t-SNE grid."))
else:
    display(Markdown("t-SNE grid skipped because `RUN_TSNE=False`."))

## KDE/ECDF Feature Distributions

KDE overlays are used only when SciPy can fit non-degenerate densities. Otherwise, the figure falls back to ECDF overlays and is titled accordingly.


In [ ]:
def ecdf_values(summary: Mapping[str, Any], feature: str) -> tuple[list[float], list[float]] | None:
    feature_payload = summary.get("features", {}).get(feature, {})
    ecdf = feature_payload.get("ecdf") if isinstance(feature_payload, Mapping) else None
    if not isinstance(ecdf, Mapping):
        return None
    x_values = ecdf.get("x")
    y_values = ecdf.get("y")
    if isinstance(x_values, list) and isinstance(y_values, list):
        return x_values, y_values
    return None


def kde_values(summary: Mapping[str, Any], feature: str) -> tuple[list[float], list[float]] | None:
    feature_payload = summary.get("features", {}).get(feature, {})
    kde = feature_payload.get("kde") if isinstance(feature_payload, Mapping) else None
    if not isinstance(kde, Mapping):
        return None
    x_values = kde.get("x")
    density = kde.get("density")
    if isinstance(x_values, list) and isinstance(density, list):
        return x_values, density
    return None


def plot_distribution_axis(
    axis: Any, panel: Mapping[str, Any], feature: str, *, prefer_kde: bool
) -> bool:
    real_summary = kde_or_ecdf_summary(panel["real_features"])
    generated_summary = kde_or_ecdf_summary(panel["generated_features"])
    plotted_kde = False
    if prefer_kde:
        real_kde = kde_values(real_summary, feature)
        generated_kde = kde_values(generated_summary, feature)
        if real_kde is not None and generated_kde is not None:
            axis.plot(real_kde[0], real_kde[1], label="real", linewidth=2.0)
            axis.plot(generated_kde[0], generated_kde[1], label=panel["family"], linewidth=1.5)
            axis.set_ylabel("density")
            plotted_kde = True
    if not plotted_kde:
        real_ecdf = ecdf_values(real_summary, feature)
        generated_ecdf = ecdf_values(generated_summary, feature)
        if real_ecdf is not None:
            axis.plot(real_ecdf[0], real_ecdf[1], label="real", linewidth=2.0)
        if generated_ecdf is not None:
            axis.plot(generated_ecdf[0], generated_ecdf[1], label=panel["family"], linewidth=1.5)
        axis.set_ylabel("ECDF")
    axis.set_xlabel(feature)
    return plotted_kde


distribution_path = REPORT_OUTPUT_DIR / "kde_ecdf_grid.png"
if RUN_KDE and panel_records:
    selected_panels = panel_records
    feature_names = []
    for feature in FEATURE_ORDER:
        if any(feature in panel["real_features"].feature_names for panel in selected_panels):
            feature_names.append(feature)
    figure, axes = plt.subplots(
        len(selected_panels),
        len(feature_names),
        figsize=(3.5 * len(feature_names), 3.1 * len(selected_panels)),
        squeeze=False,
        constrained_layout=True,
    )
    kde_count = 0
    ecdf_count = 0
    for row_index, panel in enumerate(selected_panels):
        for col_index, feature in enumerate(feature_names):
            axis = axes[row_index][col_index]
            if feature not in panel["real_features"].feature_names:
                axis.set_axis_off()
                continue
            plotted_kde = plot_distribution_axis(
                axis,
                panel,
                feature,
                prefer_kde=SCIPY_AVAILABLE,
            )
            kde_count += int(plotted_kde)
            ecdf_count += int(not plotted_kde)
            if row_index == 0:
                axis.set_title(feature)
            if col_index == 0:
                axis.text(
                    -0.35,
                    0.5,
                    f"{panel['experiment']}\n{panel['family']}",
                    transform=axis.transAxes,
                    rotation=90,
                    va="center",
                    ha="center",
                )
            if row_index == 0 and col_index == 0:
                axis.legend(frameon=False, fontsize="x-small")
    title = "KDE feature overlays" if kde_count > 0 else "ECDF feature overlays"
    if ecdf_count > 0 and kde_count > 0:
        title = "KDE/ECDF feature overlays"
    figure.suptitle(title, y=1.01)
    figure.savefig(distribution_path)
    plt.close(figure)
    display(
        pd.DataFrame([
            {
                "figure": relative_to_repo(distribution_path),
                "kde_panels": kde_count,
                "ecdf_fallback_panels": ecdf_count,
            }
        ])
    )
elif RUN_KDE:
    display(Markdown("No local model comparisons were available for the KDE/ECDF grid."))
else:
    display(Markdown("KDE/ECDF grid skipped because `RUN_KDE=False`."))

## Summary Tables

The tables below combine registry metadata, optional local summary JSON files, feature moments, and missing-output notes. Local summary metrics are opportunistic: paths without a nearby summary JSON remain valid for plotting if their tensor payload is available.


In [ ]:
registry_metric_rows = []
for record in model_records:
    selected = record.get("registered_model")
    registry_metrics = getattr(selected, "metrics", {}) if selected is not None else {}
    summary_metrics = load_summary_metrics(record.get("summary_path"))
    keys = sorted(set(registry_metrics) | set(summary_metrics))
    for key in keys:
        registry_metric_rows.append({
            "experiment": record["experiment"],
            "family": record["family"],
            "candidate_id": record.get("candidate_id"),
            "metric": key,
            "registry_value": registry_metrics.get(key),
            "local_summary_value": summary_metrics.get(key),
            "summary_path": relative_to_repo(record.get("summary_path")),
        })

metrics_table = pd.DataFrame(registry_metric_rows)
display(Markdown("### Available models and batch paths"))
display(batch_table)
display(Markdown("### Selected registry and local summary metrics"))
display(metrics_table.head(80) if not metrics_table.empty else metrics_table)
display(Markdown("### Feature means and standard deviations"))
display(feature_stats_table)
display(Markdown("### Missing outputs and notes"))
display(
    missing_outputs_table if not missing_outputs_table.empty else pd.DataFrame(columns=["note"])
)

## Interpretation Notes

- t-SNE is qualitative only. It is useful for visualising coarse feature-space separation, but it should not be treated as a model-selection metric.
- KDE/ECDF panels are more directly interpretable for financial feature retention because they compare named quantities such as terminal return, realised volatility, drawdown, autocorrelation, and jump-tail features.
- Model selection should continue to rely on quantitative metrics from `trained_models/model_registry.yaml`, model cards, and controlled evaluation reports.
- S&P500/VIX and Hawkes/SVMHJD answer different questions. S&P500/VIX targets empirical market windows with regime or volatility conditioning. Hawkes/SVMHJD targets synthetic rare-event, self-exciting jump behaviour.
- Missing local outputs are expected on clean checkouts. Use `BATCH_PATH_OVERRIDES` to point this notebook at local batches, or run the established evaluation workflows outside this notebook when new generated samples are required.
